# SIC8 — 05 Historical ML Baseline (Option C)

**Goal:** Train a reproducible supervised Logistic Regression baseline on published Silver tables and evaluate it on future, unseen years. This is an **experiment**, not a fraud-decision service.

**Policy:** Source labels are used only for training and evaluation, never as model features. No synthetic transactions, no writes, no threshold selection on the final test period. Follow the cells in order.

## 01 — Environment & experiment settings
Check Spark ML support before any expensive scan. Databricks **Serverless Environment v4** supports `pyspark.ml`; select it in the notebook's Environment panel if required.

In [0]:
from pyspark.sql import functions as F
import math
import time

SILVER = "workspace.sic8_silver"
TRANSACTIONS = f"{SILVER}.historical_transactions"
LABELS = f"{SILVER}.historical_ground_truth"
EXPECTED_TRANSACTIONS = 24_386_900
NEGATIVE_TRAIN_FRACTION = 0.03  # Sample only normal TRAIN rows for lower compute cost
F_BETA = 2.0                     # Validation objective favors recall, still penalizes false alerts
ENABLE_WRITE = False

try:
    from pyspark.ml import Pipeline
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
    from pyspark.ml.functions import vector_to_array
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
except Exception as exc:
    raise RuntimeError(
        "Spark ML is unavailable. On Databricks Serverless, select Environment version 4 "
        "and restart this notebook; do not install random versions of pyspark."
    ) from exc

print("Spark:", spark.version)
print("Transactions:", TRANSACTIONS)
print("Labels:", LABELS)
print("Writes enabled:", ENABLE_WRITE)

Spark: 4.2.0
Transactions: workspace.sic8_silver.historical_transactions
Labels: workspace.sic8_silver.historical_ground_truth
Writes enabled: False


## 02 — Read the published Silver and isolated labels
Use the exact column contract. Validate one-to-one label joins before training; a wrong join would invalidate every metric.

In [0]:
transactions = spark.table(TRANSACTIONS).select(
    "transaction_id", "source_event_time_local", "amount",
    "currency", "payment_method", "mcc_key"
)
labels = spark.table(LABELS).select(
    "transaction_id", "ground_truth_label"
)

required = {"transaction_id", "source_event_time_local", "amount", "currency", "payment_method", "mcc_key"}
assert required.issubset(set(transactions.columns))

joined = transactions.join(labels, "transaction_id", "inner")
summary = joined.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("unique_ids"),
    F.sum(F.when(F.col("ground_truth_label") == True, 1).otherwise(0)).alias("fraud_rows"),
    F.sum(F.when(F.col("ground_truth_label").isNull(), 1).otherwise(0)).alias("missing_labels"),
    F.sum(F.when(F.col("amount").isNull() | F.col("source_event_time_local").isNull(), 1).otherwise(0)).alias("bad_inputs"),
    F.sum(F.when(F.col("currency") != "USD", 1).otherwise(0)).alias("non_usd")
).first().asDict()

print("Silver audit:", summary)
assert summary["rows"] == EXPECTED_TRANSACTIONS, "Unexpected joined row count"
assert summary["unique_ids"] == summary["rows"], "Transaction ID duplicates after label join"
assert summary["missing_labels"] == 0 and summary["bad_inputs"] == 0
assert summary["non_usd"] == 0, "This experimental baseline assumes USD historical transactions"

# Keep the label for supervision, but EXCLUDE it from all feature columns later.
base = joined.withColumn("label", F.col("ground_truth_label").cast("double"))

Silver audit: {'rows': 24386900, 'unique_ids': 24386900, 'fraud_rows': 29757, 'missing_labels': 0, 'bad_inputs': 0, 'non_usd': 0}


## 03 — Honest chronological split
Determine year boundaries using **transaction volumes only**. Never choose the split based on fraud detection results. Reserve the last years as a blind test set.

In [0]:
year_counts = (base
    .withColumn("event_year", F.year("source_event_time_local"))
    .groupBy("event_year")
    .agg(F.count("*").alias("rows"), F.sum("label").alias("fraud"))
    .orderBy("event_year")
    .collect())

year_distribution = [(int(row["event_year"]), int(row["rows"]), int(row["fraud"])) for row in year_counts]
print("Year distribution (year, rows, fraud):")
for item in year_distribution:
    print(item)

if len(year_distribution) < 3:
    raise ValueError("Need at least 3 distinct years for chronological train/validation/test")

total = sum(row[1] for row in year_distribution)
cumulative = 0
train_end_year = None
validation_end_year = None
for year, count, _ in year_distribution:
    cumulative += count
    if train_end_year is None and cumulative >= 0.70 * total:
        train_end_year = year
    if validation_end_year is None and cumulative >= 0.85 * total:
        validation_end_year = year

max_year = year_distribution[-1][0]
if not (train_end_year < validation_end_year < max_year):
    raise ValueError("Automatic split leaves no distinct validation/test years; choose valid year boundaries manually")

print("Chronological boundaries: TRAIN <=", train_end_year,
      "| VALIDATION <=", validation_end_year,
      "| TEST >", validation_end_year)

Year distribution (year, rows, fraud):
(1991, 1585, 0)
(1992, 5134, 0)
(1993, 8378, 0)
(1994, 14316, 0)
(1995, 20928, 0)
(1996, 29945, 10)
(1997, 49753, 32)
(1998, 78345, 32)
(1999, 118250, 24)
(2000, 177729, 171)
(2001, 257998, 354)
(2002, 350732, 139)
(2003, 466408, 311)
(2004, 597003, 620)
(2005, 746653, 229)
(2006, 908793, 1118)
(2007, 1064483, 1881)
(2008, 1223460, 3710)
(2009, 1355434, 1140)
(2010, 1491225, 3835)
(2011, 1570551, 55)
(2012, 1610829, 1333)
(2013, 1650917, 2018)
(2014, 1672343, 1052)
(2015, 1701371, 3281)
(2016, 1708924, 3579)
(2017, 1723360, 255)
(2018, 1721615, 2491)
(2019, 1723938, 2087)
(2020, 336500, 0)
Chronological boundaries: TRAIN <= 2015 | VALIDATION <= 2018 | TEST > 2018


## 04 — Build point-in-time features from transactions
Only use information present on the transaction itself. In this starter model, we deliberately exclude fraud labels, outcome/errors, demographic snapshots, customer IDs, and post-transaction statistics.

In [0]:
features = (base
    .withColumn("event_year", F.year("source_event_time_local"))
    .withColumn("hour", F.hour("source_event_time_local").cast("double"))
    .withColumn("weekday", F.dayofweek("source_event_time_local").cast("double"))
    .withColumn("log_abs_amount", F.log1p(F.abs(F.col("amount").cast("double"))))
    .withColumn("is_refund", F.when(F.col("amount") < 0, 1.0).otherwise(0.0))
    .withColumn("is_online", F.when(F.col("payment_method") == "ONLINE", 1.0).otherwise(0.0))
    .withColumn("is_swipe", F.when(F.col("payment_method") == "SWIPE", 1.0).otherwise(0.0))
    .withColumn("is_chip", F.when(F.col("payment_method") == "CHIP", 1.0).otherwise(0.0))
    .withColumn("sin_hour", F.sin(F.col("hour") * F.lit(2 * math.pi / 24)))
    .withColumn("cos_hour", F.cos(F.col("hour") * F.lit(2 * math.pi / 24)))
    .withColumn("mcc_code", F.coalesce(F.col("mcc_key"), F.lit("UNKNOWN")))
    .select("transaction_id", "event_year", "label", "log_abs_amount", "is_refund", "is_online",
            "is_swipe", "is_chip", "weekday", "sin_hour", "cos_hour", "mcc_code")
)

feature_columns = ["log_abs_amount", "is_refund", "is_online", "is_swipe", "is_chip",
                   "weekday", "sin_hour", "cos_hour", "mcc_onehot"]
assert "label" not in feature_columns
assert "ground_truth_label" not in feature_columns
assert "transaction_id" not in feature_columns
print("Numeric features:", feature_columns[:-1], "| Categorical: MCC")

Numeric features: ['log_abs_amount', 'is_refund', 'is_online', 'is_swipe', 'is_chip', 'weekday', 'sin_hour', 'cos_hour'] | Categorical: MCC


## 05 — Split first; sample training negatives only
All fraud-labelled TRAIN rows are retained. Sample normal TRAIN rows deterministically to limit training cost. Validation and test remain **complete**, preserving the real class imbalance.

In [0]:
train_full = features.filter(F.col("event_year") <= train_end_year)
validation = features.filter((F.col("event_year") > train_end_year) & (F.col("event_year") <= validation_end_year))
test = features.filter(F.col("event_year") > validation_end_year)

sample_limit = int(NEGATIVE_TRAIN_FRACTION * 1_000_000)
train_sampled = train_full.filter(
    (F.col("label") == 1.0) |
    (F.pmod(F.xxhash64(F.col("transaction_id")), F.lit(1_000_000)) < F.lit(sample_limit))
).withColumn(
    "sample_weight", F.when(F.col("label") == 1.0, F.lit(1.0))
     .otherwise(F.lit(1.0 / NEGATIVE_TRAIN_FRACTION))
)

# Audits are intentionally before fitting, including the complete holdout population.
for name, frame in [("TRAIN_SAMPLED", train_sampled), ("VALIDATION_FULL", validation), ("TEST_FULL", test)]:
    result = frame.agg(F.count("*").alias("rows"), F.sum("label").alias("fraud")).first().asDict()
    print(name, result)
    assert result["rows"] > 0 and 0 < result["fraud"] < result["rows"], f"Invalid class distribution in {name}"

print("Train only: negative inclusion rate =", NEGATIVE_TRAIN_FRACTION)
print("Training sampling weights correct the negative inclusion probability (not a full probability calibration guarantee).")

TRAIN_SAMPLED {'rows': 536090, 'fraud': 21345.0}
VALIDATION_FULL {'rows': 5153899, 'fraud': 6325.0}
TEST_FULL {'rows': 2060438, 'fraud': 2087.0}
Train only: negative inclusion rate = 0.03
Training sampling weights correct the negative inclusion probability (not a full probability calibration guarantee).


## 06 — Train a supervised Logistic Regression baseline
`StringIndexer` and `OneHotEncoder` are fitted on **TRAIN only**, so category handling cannot learn from validation/test. The model trains using actual historical fraud labels.

In [0]:
start_train = time.perf_counter()
stages = [
    StringIndexer(inputCol="mcc_code", outputCol="mcc_index", handleInvalid="keep"),
    OneHotEncoder(inputCols=["mcc_index"], outputCols=["mcc_onehot"], handleInvalid="keep"),
    VectorAssembler(inputCols=feature_columns, outputCol="features"),
    LogisticRegression(featuresCol="features", labelCol="label", weightCol="sample_weight",
                       maxIter=35, regParam=0.05, elasticNetParam=0.0, standardization=True)
]

ml_pipeline = Pipeline(stages=stages)
ml_model = ml_pipeline.fit(train_sampled)
print("Training complete; seconds:", round(time.perf_counter() - start_train, 2))
print("No data was written and no model was deployed.")

Training complete; seconds: 34.17
No data was written and no model was deployed.


## 07 — Validation predictions and threshold selection
Scores come from the trained model. Evaluate threshold candidates on the **full validation years**; select exactly one threshold before touching the blind test.

In [0]:
THRESHOLDS = [0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.5]

validation_scored = (ml_model.transform(validation)
    .withColumn("risk_score", vector_to_array(F.col("probability"))[1].cast("double"))
    .select("transaction_id", "label", "risk_score", "rawPrediction"))

# One aggregation evaluates every threshold without collecting individual transactions.
def evaluate_thresholds(frame, thresholds):
    expressions = [F.count("*").alias("rows"), F.sum("label").alias("actual_fraud")]
    for i, threshold in enumerate(thresholds):
        flagged = F.col("risk_score") >= F.lit(float(threshold))
        expressions += [
            F.sum(F.when(flagged, 1).otherwise(0)).alias(f"alert_{i}"),
            F.sum(F.when(flagged & (F.col("label") == 1.0), 1).otherwise(0)).alias(f"tp_{i}")
        ]
    aggregate = frame.agg(*expressions).first().asDict()
    rows = []
    for i, threshold in enumerate(thresholds):
        flagged = int(aggregate[f"alert_{i}"])
        tp = int(aggregate[f"tp_{i}"])
        fp = flagged - tp
        fn = int(aggregate["actual_fraud"]) - tp
        tn = int(aggregate["rows"]) - tp - fp - fn
        precision = tp / flagged if flagged else 0.0
        recall = tp / int(aggregate["actual_fraud"]) if aggregate["actual_fraud"] else 0.0
        denom = F_BETA * F_BETA * precision + recall
        fbeta = (1 + F_BETA * F_BETA) * precision * recall / denom if denom else 0.0
        rows.append({"threshold": float(threshold), "alerts": flagged, "tp": tp, "fp": fp,
                     "fn": fn, "tn": tn, "precision": precision, "recall": recall, "f2": fbeta})
    return rows

validation_results = evaluate_thresholds(validation_scored, THRESHOLDS)
for row in validation_results:
    print(row)

best_val = max(validation_results, key=lambda r: (r["f2"], r["precision"]))
CHOSEN_THRESHOLD = best_val["threshold"]
print("VALIDATION-SELECTED THRESHOLD:", CHOSEN_THRESHOLD)
print("This threshold is locked before running the blind test.")

{'threshold': 0.001, 'alerts': 5153899, 'tp': 6325, 'fp': 5147574, 'fn': 0, 'tn': 0, 'precision': 0.0012272262223221682, 'recall': 1.0, 'f2': 0.00610615656977073}
{'threshold': 0.002, 'alerts': 13371, 'tp': 746, 'fp': 12625, 'fn': 5579, 'tn': 5134949, 'precision': 0.05579238650811458, 'recall': 0.11794466403162056, 'f2': 0.09645470766207237}
{'threshold': 0.005, 'alerts': 5498, 'tp': 417, 'fp': 5081, 'fn': 5908, 'tn': 5142493, 'precision': 0.07584576209530738, 'recall': 0.06592885375494072, 'f2': 0.0676992012468342}
{'threshold': 0.01, 'alerts': 213, 'tp': 76, 'fp': 137, 'fn': 6249, 'tn': 5147437, 'precision': 0.3568075117370892, 'recall': 0.012015810276679842, 'f2': 0.014894367577313525}
{'threshold': 0.02, 'alerts': 213, 'tp': 76, 'fp': 137, 'fn': 6249, 'tn': 5147437, 'precision': 0.3568075117370892, 'recall': 0.012015810276679842, 'f2': 0.014894367577313525}
{'threshold': 0.03, 'alerts': 213, 'tp': 76, 'fp': 137, 'fn': 6249, 'tn': 5147437, 'precision': 0.3568075117370892, 'recall': 

## 08 — Blind future-year evaluation
Run the model once on the untouched future-year test set. Report confusion matrix, precision, recall, F2, alert volume and PR-AUC.

In [0]:
test_scored = (ml_model.transform(test)
    .withColumn("risk_score", vector_to_array(F.col("probability"))[1].cast("double"))
    .select("transaction_id", "label", "risk_score", "rawPrediction"))

test_results = evaluate_thresholds(test_scored, [CHOSEN_THRESHOLD])
test_report = test_results[0]

print("BLIND TEST REPORT")
for name, value in test_report.items():
    print(f"  {name}: {value}")

try:
    evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                               metricName="areaUnderPR")
    pr_auc = evaluator.evaluate(test_scored)
    print("  areaUnderPR:", pr_auc)
except Exception as exc:
    print("  PR-AUC unavailable in this Spark environment:", str(exc)[:220])

print("  historical fraud baseline prevalence:", 29_757 / 24_386_900)
print("  predicted fraud is NOT the original source label.")

BLIND TEST REPORT
  threshold: 0.002
  alerts: 5117
  tp: 213
  fp: 4904
  fn: 1874
  tn: 2053447
  precision: 0.04162595270666406
  recall: 0.1020603737422137
  f2: 0.07909394727070182
  areaUnderPR: 0.0068444306157312935
  historical fraud baseline prevalence: 0.0012202042900081602
  predicted fraud is NOT the original source label.


## 09 — Final experiment summary (read-only)
Compare this supervised baseline against the historical rule baseline only on **matching future-year rows**. Do not claim that Sayed's model was independently evaluated until his training periods and feature pipeline are verified.

In [0]:
print("MODEL: LogisticRegression (Spark ML, supervised)")
print("SOURCE:", TRANSACTIONS)
print("TRAIN years: <=", train_end_year, "| VALIDATION years:", train_end_year + 1,
      "to", validation_end_year, "| TEST years: >", validation_end_year)
print("Training normal-row fraction:", NEGATIVE_TRAIN_FRACTION)
print("Feature count before one-hot encoding:", len(feature_columns))
print("Selected threshold (validation only):", CHOSEN_THRESHOLD)
print("Blind test precision:", round(test_report["precision"], 5))
print("Blind test recall:", round(test_report["recall"], 5))
print("Blind test alert count:", test_report["alerts"])
print("WRITE ENABLED:", ENABLE_WRITE)
assert ENABLE_WRITE is False, "This first experiment must not publish model outputs"
print("STATUS: Experiment complete; review model reliability before any deployment")

MODEL: LogisticRegression (Spark ML, supervised)
SOURCE: workspace.sic8_silver.historical_transactions
TRAIN years: <= 2015 | VALIDATION years: 2016 to 2018 | TEST years: > 2018
Training normal-row fraction: 0.03
Feature count before one-hot encoding: 9
Selected threshold (validation only): 0.002
Blind test precision: 0.04163
Blind test recall: 0.10206
Blind test alert count: 5117
WRITE ENABLED: False
STATUS: Experiment complete; review model reliability before any deployment
